In [1]:
import sys, torch
sys.path.insert(0, "qwen3")                      # match the architecture of the checkpoint
from model import TinyQwen
from tokenizer import CharTokenizer


In [2]:
! ls

README.md  data       demo.ipynb lora       qwen3_5
acestep    deepseek3  gemma4     qwen3


In [5]:

ckpt = torch.load("qwen3/tiny_qwen.pt", map_location="cpu", weights_only=False)
tok = CharTokenizer(ckpt["chars"])
model = TinyQwen(ckpt["cfg"]); model.load_state_dict(ckpt["model"]); model.eval()

model

TinyQwen(
  (embed_tokens): Embedding(30, 32)
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (input_layernorm): RMSNorm()
      (attn): Attention(
        (q_proj): Linear(in_features=32, out_features=32, bias=False)
        (k_proj): Linear(in_features=32, out_features=16, bias=False)
        (v_proj): Linear(in_features=32, out_features=16, bias=False)
        (o_proj): Linear(in_features=32, out_features=32, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=32, out_features=64, bias=False)
        (up_proj): Linear(in_features=32, out_features=64, bias=False)
        (down_proj): Linear(in_features=64, out_features=32, bias=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=32, out_features=30, bias=False)
)

In [7]:
model.embed_tokens.weight

Parameter containing:
tensor([[-2.5946e-02, -1.6901e-01, -3.9247e-01, -6.5871e-01,  6.8593e-01,
          2.8574e-02,  8.0509e-01,  1.3043e-02,  2.2318e-01,  1.0939e+00,
         -1.2547e+00, -4.1371e-01,  2.7323e-02, -3.2181e-01, -9.3095e-01,
          1.4433e+00,  1.2705e+00, -2.8188e-01,  3.3241e-01,  7.3920e-01,
         -1.7423e+00,  4.6254e-01,  1.3124e+00,  5.9482e-01,  2.2654e-01,
         -1.3165e+00, -1.1114e+00, -3.3506e-01,  4.1676e-01, -5.5593e-01,
          1.4280e+00,  2.4521e+00],
        [-6.3700e-01, -2.9986e-01,  7.1820e-01,  2.2338e-01,  1.0102e-02,
          8.4068e-01, -1.0364e+00, -3.6484e-01, -5.1326e-01, -8.9332e-01,
          5.1901e-01, -1.3854e+00, -1.0952e+00,  4.6776e-01, -4.6867e-01,
         -6.0840e-01,  1.5581e+00, -6.8744e-01,  1.2181e+00, -2.0417e-01,
         -1.2345e+00,  1.9673e+00,  2.5926e+00, -1.5755e+00,  1.1151e+00,
         -1.0973e+00,  5.7082e-01, -2.4900e-01,  4.6837e-01,  1.5202e+00,
          1.4613e+00, -3.1769e-01],
        [-5.0040e-

In [8]:
tok.encode('a')
# 1. indiste a harfinin karsiligi varmis

[1]

In [10]:
tok.decode([0])

'\n'

In [12]:
model.embed_tokens.weight[1]
# a harfinin 30 boyutlu embed uzaydaki yeri

tensor([-0.6370, -0.2999,  0.7182,  0.2234,  0.0101,  0.8407, -1.0364, -0.3648,
        -0.5133, -0.8933,  0.5190, -1.3854, -1.0952,  0.4678, -0.4687, -0.6084,
         1.5581, -0.6874,  1.2181, -0.2042, -1.2345,  1.9673,  2.5926, -1.5755,
         1.1151, -1.0973,  0.5708, -0.2490,  0.4684,  1.5202,  1.4613, -0.3177],
       grad_fn=<SelectBackward0>)

In [4]:
start = torch.full((10, 1), tok.eos_id, dtype=torch.long)   # every name starts at EOS/newline
out = model.generate(start, max_new_tokens=model.cfg.max_seq_len,
                     temperature=0.8, eos_id=tok.eos_id)     # stops each row at EOS
out

tensor([[ 0,  2, 26,  7, 29, 17,  1,  0,  0],
        [ 0, 24,  5, 19,  9, 14,  0,  0,  0],
        [ 0,  4,  5, 21,  1, 12,  0,  0,  0],
        [ 0, 25, 23,  8,  1, 14,  0,  0,  0],
        [ 0,  9, 17,  7, 26, 12,  0,  0,  0],
        [ 0, 13,  9,  5, 12,  9,  8,  0,  0],
        [ 0,  8,  1,  0,  0,  0,  0,  0,  0],
        [ 0, 25, 23,  7, 26, 12,  0,  0,  0],
        [ 0, 18,  5,  4,  1, 19,  0,  0,  0],
        [ 0,  1, 12, 16,  5, 17,  5, 14,  0]])

In [5]:
for row in out.tolist():
    print(tok.decode(row[1:]).split("\n")[0])

bügşra
çetin
deval
özhan
irgül
mielih
ha
özgül
sedat
alperen
